# 2) MTG Audio Extraction
- A pipeline to extract MTG audio data
- All that is required is a CLIENT_ID to pull the pipeline

## Library Imports

In [ ]:
import os, time, random, subprocess
import requests
import pandas as pd
from tqdm import tqdm

import os, glob

import zipfile, os, glob

## Pipeline

In [ ]:
# Pipline customized for Jamedo songs

CLIENT_ID = # get from Jamendo developer portal

GENRES = ["rock", "pop", "electronic", "hiphop", "jazz",
          "classical", "folk", "metal", "reggae", "blues"]

SONGS_PER_GENRE = 10 # Song selection
N_TRACKS = len(GENRES) * SONGS_PER_GENRE

OUT_DIR = "./jamendo_balanced_" # Name of folder
SEED = 42
random.seed(SEED)

os.makedirs(OUT_DIR, exist_ok=True)

BASE = "https://api.jamendo.com/v3.0/tracks"
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "DepthExternship/1.0 (python requests)"})

def get_tracks_page(genre, limit=200, offset=0):
    params = {
        "client_id": CLIENT_ID,
        "format": "json",
        "limit": limit,
        "offset": offset,
        "tags": genre,                  # <-- single genre per request
        "audioformat": "mp32",
        "audiodlformat": "mp32",
        "durationbetween": "120_900",
    }
    r = SESSION.get(BASE, params=params, timeout=30)
    r.raise_for_status()
    j = r.json()
    headers = j.get("headers", {})
    if headers.get("status") and headers.get("status") != "success":
        raise RuntimeError(f"Jamendo API error: {headers}")
    return j.get("results", [])

def download(url, path):
    with SESSION.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

rows = []
seen_ids = set()

pbar = tqdm(total=N_TRACKS, desc="Downloading balanced genres")

for genre in GENRES:
    genre_dir = os.path.join(OUT_DIR, genre)
    os.makedirs(genre_dir, exist_ok=True)

    saved_for_genre = 0
    offset = 0

    while saved_for_genre < SONGS_PER_GENRE:
        results = get_tracks_page(genre, limit=200, offset=offset)
        if not results:
            break

        random.shuffle(results)

        for tr in results:
            tid = tr.get("id")
            if not tid or tid in seen_ids:
                continue

            # prefer audiodownload; fall back to audio stream
            url = tr.get("audiodownload") if tr.get("audiodownload_allowed") else None
            if not url:
                url = tr.get("audio")
            if not url:
                continue

            filename = f"{saved_for_genre+1:02d}_{tid}.mp3"
            rel_file = os.path.join(genre, filename)
            abs_path = os.path.join(OUT_DIR, rel_file)

            try:
                download(url, abs_path)
            except Exception:
                continue

            seen_ids.add(tid)
            rows.append({
                "genre": genre,
                "file": rel_file,  # includes genre/...
                "track_id": tid,
                "name": tr.get("name"),
                "artist_name": tr.get("artist_name"),
                "duration": tr.get("duration"),
                "tags": tr.get("tags"),
                "audiodownload_allowed": tr.get("audiodownload_allowed"),
                "license_ccurl": tr.get("license_ccurl"),
                "source_url": url,
            })

            saved_for_genre += 1
            pbar.update(1)

            if saved_for_genre >= SONGS_PER_GENRE:
                break

        offset += 200
        time.sleep(0.25)

pbar.close()

df = pd.DataFrame(rows)
df.to_csv(os.path.join(OUT_DIR, "manifest_balanced.csv"), index=False)

print("Downloaded:", len(df))
print(df.groupby("genre").size())
df.sample(3)

## Zip Files

### jamendo_balanced_50

In [ ]:
# checking size

OUT_DIR = "./jamendo_balanced_50"
zip_path = "jamendo_balanced_50.zip"

files = glob.glob(os.path.join(OUT_DIR, "**", "*.mp3"), recursive=True)
files += glob.glob(os.path.join(OUT_DIR, "*.csv"))

print("MP3 found:", len(files))

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f, arcname=os.path.relpath(f, OUT_DIR))

print("Created:", os.path.abspath(zip_path))

In [ ]:
# ziping files to download

OUT_DIR = "./jamendo_balanced_50"
zip_path = "./jamendo_balanced_50.zip" # name of zip file

files = glob.glob(os.path.join(OUT_DIR, "*.mp3")) + [os.path.join(OUT_DIR, "manifest.csv")]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for f in files:
        if os.path.exists(f):
            z.write(f, arcname=os.path.relpath(f, OUT_DIR))

print("Created:", zip_path)

### jamendo_balanced_200

In [ ]:
# checking size

OUT_DIR = "./jamendo_balanced_200"
zip_path = "jamendo_balanced_200.zip"

files = glob.glob(os.path.join(OUT_DIR, "**", "*.mp3"), recursive=True)
files += glob.glob(os.path.join(OUT_DIR, "*.csv"))

print("MP3 found:", len(files))

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f, arcname=os.path.relpath(f, OUT_DIR))

print("Created:", os.path.abspath(zip_path))

In [ ]:
# ziping files to download

OUT_DIR = "./jamendo_balanced_200"
zip_path = "./jamendo_balanced_200.zip" # name of zip file

files = glob.glob(os.path.join(OUT_DIR, "*.mp3")) + [os.path.join(OUT_DIR, "manifest.csv")]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for f in files:
        if os.path.exists(f):
            z.write(f, arcname=os.path.relpath(f, OUT_DIR))

print("Created:", zip_path)

### jamendo_balance_1000
- Could not extract, lost file

In [ ]:
# checking size

OUT_DIR = "./jamendo_balanced_1000"
zip_path = "jamendo_balanced_1000.zip"

files = glob.glob(os.path.join(OUT_DIR, "**", "*.mp3"), recursive=True)
files += glob.glob(os.path.join(OUT_DIR, "*.csv"))

print("MP3 found:", len(files))

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f, arcname=os.path.relpath(f, OUT_DIR))

print("Created:", os.path.abspath(zip_path))

In [ ]:
# ziping files to download

OUT_DIR = "./jamendo_balanced_1000"
zip_path = "./jamendo_balanced_1000.zip" # name of zip file

files = glob.glob(os.path.join(OUT_DIR, "*.mp3")) + [os.path.join(OUT_DIR, "manifest.csv")]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for f in files:
        if os.path.exists(f):
            z.write(f, arcname=os.path.relpath(f, OUT_DIR))

print("Created:", zip_path)